# Implementing an SLR-Table-Generator

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.

## A Grammar for Grammars

We use the following grammar to describe the syntax of a context free grammar.
Since <span style="font-variant:small-caps;">Lark</span> supports *EBNF*, the Python version can use the postfix operators
`*` (zero or more repetitions) and `+` (one or more repetitions) and it can group symbols with parentheses.
In *EBNF*, the grammar reads as follows:
```
grammar : rule+
rule    : VARIABLE ':' body ('|' body)* ';'
body    : item*
item    : VARIABLE | TOKEN | LITERAL
```
Menhir grammars are written in *BNF*, so the repetitions are expressed by recursive rules below.
Here, `VARIABLE`, `TOKEN`, and `LITERAL` are *terminals*:
- a `VARIABLE` is a *syntactical variable* and is written in lower case letters,
- a `TOKEN` is the name of a token and is written in upper case letters, while
- a `LITERAL` is a string enclosed in single quotes.

As the goal is to generate an *SLR-table-generator* we first need to implement a parser for context free grammars.
The file `arith.g` in the directory `Examples` contains an example grammar that describes arithmetic expressions.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree
;;
cat "Examples/arith.g"

We use the scanner generator `ocamllex` and the parser generator `Menhir` to develop a parser for context free grammars.
(The Python version of this notebook uses <span style="font-variant:small-caps;">Lark</span>.)  The scanner and the
parser are specified separately: the scanner specification defines the tokens via regular expressions, while the
grammar defines the structure.  The function `generate_parser` and the module `Parse_tree`, which describes parse
trees in the style of <span style="font-variant:small-caps;">Lark</span>, are provided by the file
`../lib/parser_generator.ml`, which has been loaded in the previous cell.

In [ ]:
(* generate_parser and the module Parse_tree have been loaded at the beginning of the notebook *)

The scanner specification `grammar_lexer` and the grammar `grammar_of_grammars` below describe context free grammars.
The following conventions are important:
- The scanner specification defines the tokens `VARIABLE`, `TOKEN`, and `LITERAL` via regular expressions.  In order
  to be able to print all tokens, the tokens `COLON`, `VBAR`, and `SEMICOLON` also carry a value of type
  `Parse_tree.tok`.  These are the names that <span style="font-variant:small-caps;">Lark</span> uses for the strings
  `":"`, `"|"`, and `";"`.
- The scanner discards white space and comments.  A comment starts with the string `//` and extends to the end of the line.
- In the grammar, every rule is followed by a *semantic action*, which builds the parse tree.  The strings `":"`,
  `"|"`, and `";"` do not show up in the parse tree, since the actions do not mention them.
- In <span style="font-variant:small-caps;">Lark</span>, the question mark in `?item` *inlines* the rule `item`.  In
  Menhir, the action `{ Token v }` simply returns the token, so the variables, tokens, and literals show up as immediate
  children of the node `body`.
- <span style="font-variant:small-caps;">Lark</span> supports the operators `*` and `+` of *EBNF*.  Menhir's grammar
  below uses the recursive rules `rule_list`, `more_bodies`, and `item_list` instead, which return OCaml lists.

In [ ]:
let grammar_lexer = {|
{
  open Grammar_parser
}

rule token = parse
  | [' ' '\t' '\r' '\012']+                 { token lexbuf }      (* ignore white space *)
  | '\n'                                    { Lexing.new_line lexbuf; token lexbuf }
  | "//" [^ '\n']*                          { token lexbuf }      (* ignore comments *)
  | ':'                                     { COLON (Parse_tree.token "COLON" lexbuf) }
  | '|'                                     { VBAR (Parse_tree.token "VBAR" lexbuf) }
  | ';'                                     { SEMICOLON (Parse_tree.token "SEMICOLON" lexbuf) }
  | ['a'-'z'] ['a'-'z' '0'-'9' '_']*        { VARIABLE (Parse_tree.token "VARIABLE" lexbuf) }
  | ['A'-'Z'] ['A'-'Z' '0'-'9' '_']*        { TOKEN (Parse_tree.token "TOKEN" lexbuf) }
  | '\'' [^ '\'' '\n']+ '\''   { LITERAL (Parse_tree.token "LITERAL" lexbuf) }
  | eof                                     { EOF }
  | _                                       { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar_of_grammars = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> VARIABLE TOKEN LITERAL COLON VBAR SEMICOLON
%token EOF

%start <Parse_tree.tree> grammar
%type  <Parse_tree.tree> rule body item
%type  <Parse_tree.tree list> rule_list more_bodies item_list

%%

grammar: rs = rule_list EOF                     { Tree ("grammar", rs) }    // a grammar is a non-empty list of rules

rule_list: r = rule                             { [r] }
         | r = rule rs = rule_list              { r :: rs }

rule: v = VARIABLE COLON b = body bs = more_bodies SEMICOLON
                                                { Tree ("rule", Token v :: b :: bs) }

more_bodies: /* empty */                        { [] }                     // the bodies are separated by "|"
           | VBAR b = body bs = more_bodies     { b :: bs }

body: items = item_list                         { Tree ("body", items) }   // a body can be empty

item_list: /* empty */                          { [] }
         | i = item is = item_list              { i :: is }

item: v = VARIABLE                              { Token v }
    | t = TOKEN                                 { Token t }
    | l = LITERAL                               { Token l }
|}

The function `generate_parser` creates the scanner and the parser.  It runs `ocamllex` and `menhir --lalr`, which
creates an *LALR(1) parser*.  This works because the grammar given above happens to be an *LALR(1) grammar*.  The
scanner `Grammar_lexer.token` can also be called on its own.  This enables us to inspect the token stream via the
function `lex` that is used in the function `test_scanner` below.

In [ ]:
generate_parser ~name:"grammar" ~lexer:grammar_lexer ~grammar:grammar_of_grammars

Given the name of a file containing a grammar, the function `test_scanner` prints the tokens that
the scanner produces.  Every token `t` provides the fields `t.kind`, `t.value`, `t.line`, and `t.column`.  The function
`lex` calls the scanner until it returns the token `EOF` and collects the tokens.

In [ ]:
let lex (program : string) : tok list =
  let lexbuf = Lexing.from_string program in
  let rec loop () =
    match Grammar_lexer.token lexbuf with
    | Grammar_parser.EOF -> []
    | VARIABLE t | TOKEN t | LITERAL t | COLON t | VBAR t | SEMICOLON t -> t :: loop ()
  in
  loop ()

let test_scanner (file_name : string) : unit =
  let program = read_file file_name in
  print_endline program;
  List.iter (fun t -> Printf.printf "line %2d, column %2d: %-10s%s\n%!" t.line t.column t.kind t.value) (lex program)

In [ ]:
test_scanner "Examples/arith.g"

The function `Parse_tree.parse_string` returns a *parse tree*.  The function `pretty` renders this tree as an indented
string, like the method `pretty` of <span style="font-variant:small-caps;">Lark</span>.  Note that the strings `":"`,
`"|"`, and `";"` do not occur in this tree and that there are no nodes labelled `item`.

In [ ]:
print_string (pretty (parse_string Grammar_parser.grammar Grammar_lexer.token (read_file "Examples/arith.g")));;
flush stdout

The parse tree is not a convenient representation for the generation of an *SLR-table*.  Therefore,
we transform it into a list of rules, where every rule is a list of strings.  Below, we define a few *type abbreviations*
in order to make the types of this transformation more readable.  (In Python, a rule is a tuple.  OCaml has no tuples
of variable length, so we use lists instead.)

In [ ]:
type variable = string
type token    = string
type symbol   = string
type symbols  = symbol list

In Python, the class `GrammarTransformer` converts the parse tree into this representation.  For every grammar
rule of the grammar of grammars there is a method of the same name in this class.  When
<span style="font-variant:small-caps;">Lark</span> transforms the parse tree, it works bottom up: the method
that is named after a rule is called with the list of the already transformed children of the corresponding
node.  In OCaml, the module `GrammarTransformer` contains a function for every rule, and the function `transform`
traverses the parse tree bottom up and calls these functions.  The functions return the following types:
```
body:    symbols
rule:    symbols list
grammar: symbols list
```
Note that the function `rule` returns a *list* of rules because a single rule of the form
`v : b1 | b2 | b3 ;` defines three grammar rules.  The function `grammar` therefore has to flatten the list of
these lists.

In [ ]:
module GrammarTransformer = struct
  let body (items : tok list) : symbols = List.map (fun (item : tok) -> item.value) items

  let rule (variable : tok) (bodies : symbols list) : symbols list =
    List.map (fun body -> variable.value :: body) bodies

  let grammar (rules : symbols list list) : symbols list = List.concat rules

  let transform (tree : tree) : symbols list =
    let tokens children = List.map (function Token t -> t | Tree _ -> invalid_arg "token expected") children in
    let transform_body = function
      | Tree ("body", items) -> body (tokens items)
      | _ -> invalid_arg "body expected"
    in
    let transform_rule = function
      | Tree ("rule", Token variable :: bodies) -> rule variable (List.map transform_body bodies)
      | _ -> invalid_arg "rule expected"
    in
    match tree with
    | Tree ("grammar", rules) -> grammar (List.map transform_rule rules)
    | _ -> invalid_arg "grammar expected"
end

The function `parse` takes the name of a file containing a grammar and returns the list of the
grammar rules that are defined in this file.  If the file contains a syntax error, the scanner or the parser raises
the exception `Unexpected_input`, which corresponds to the exception `UnexpectedInput` of
<span style="font-variant:small-caps;">Lark</span>.  It knows the line and the column of the error, while the
function `get_context` returns the offending line together with a marker pointing at the error.  After printing the
error message, the exception is raised again.

In [ ]:
let parse (file : string) : symbols list =
  let grammar = read_file file in
  display_text grammar;
  let parse_tree =
    try parse_string Grammar_parser.grammar Grammar_lexer.token grammar
    with Unexpected_input { line; column; position; _ } as error ->
      display_text (Printf.sprintf "Syntax error in line %d, column %d:\n%s" line column (get_context grammar position));
      raise error
  in
  GrammarTransformer.transform parse_tree

In [ ]:
let ruleList = parse "Examples/arith.g"
;;
ruleList

The parser will return a list of grammar rules, where each rule of the form
$$ a \rightarrow \beta $$
is stored as the list `[a] @ 𝛽`.

The type abbreviations defined above make the types more readable.

In [ ]:
#show_type symbols

### The Module `GrammarRule`

In [ ]:
(* Python needs the module dataclasses here.  In OCaml, records can be used as keys of dictionaries immediately. *)

The module `GrammarRule` is used to store a single grammar rule.  As we have to use grammar rules as *keys* in a
dictionary later, the Python class has to provide the methods `__eq__`, `__ne__`, and `__hash__`.  In OCaml, records
are compared *structurally*, so this is not necessary.  The function `to_string` corresponds to the method `__repr__`;
it is installed as the printer of grammar rules.

In [ ]:
module GrammarRule = struct
  type t = { mVariable : variable; mBody : symbols }

  let make (variable : variable) (body : symbols) : t = { mVariable = variable; mBody = body }

  let to_string (self : t) : string = Printf.sprintf "%s → %s" self.mVariable (String.concat " " self.mBody)

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer GrammarRule.pp

The function `transform rules` takes a list of lists representing grammar rules and converts these lists into
grammar rules of type `GrammarRule.t`.

In [ ]:
let transform (rules : symbols list) : GrammarRule.t list =
  List.map (fun rule -> GrammarRule.make (List.hd rule) (List.tl rule)) rules

In [ ]:
let ruleList : symbols list = ruleList

In [ ]:
ruleList

In [ ]:
let grammar = transform ruleList

In [ ]:
grammar

Given a string `name`, which is either a *variable*, a *token*, or a *literal*, the function `is_var` checks whether
`name` is a variable.  The function can distinguish variable names from tokens and literals because variable names
consist only of lower case letters, while tokens are all uppercase and literals start with the character "`'`".  The
only variable that does not start with an ASCII letter is the variable `ŝ` of the augmented grammar.

In [ ]:
let is_var (name : symbol) : bool =
  match name.[0] with
  | 'a' .. 'z' -> true
  | c -> Char.code c >= 128   (* the variable ŝ *)

Given a list `rules` of grammar rules, the function `collect_variables rules` returns the set of all *variables* occuring in `rules`.

In [ ]:
let collect_variables (rules : GrammarRule.t list) : variable FSet.t =
  FSet.of_list (List.map (fun (rule : GrammarRule.t) -> rule.mVariable) rules)

In [ ]:
collect_variables grammar

Given a list `rules` of grammar rules, the function `collect_tokens rules` returns the set of all *tokens* and *literals* occuring in `rules`.

In [ ]:
let collect_tokens (rules : GrammarRule.t list) : token FSet.t =
  FSet.of_list (List.concat_map (fun (rule : GrammarRule.t) -> List.filter (fun item -> not (is_var item)) rule.mBody) rules)

In [ ]:
collect_tokens grammar

## Marked Rules

The module `MarkedRule` stores a single *marked rule* of the form
$$ v \rightarrow \alpha \bullet \beta $$
where the *variable* $v$ is stored in the field `mVariable`, while $\alpha$ and $\beta$ are stored in the fields `mAlpha`and `mBeta` respectively.  These fields contain lists of *grammar symbols*.  A *grammar symbol* is either
- a *variable*,
- a *token*, or
- a *literal*, i.e. a string enclosed in single quotes.


Later, we need to maintain sets of *marked rules* to represent *states*.  In Python, the class therefore has to define
the methods `__eq__`, `__ne__`, and `__hash__`.  In OCaml, records are compared structurally, so marked rules can be
stored in sets of type `FSet.t` immediately.

In [ ]:
module MarkedRule = struct
  type t = { mVariable : variable; mAlpha : symbols; mBeta : symbols }

  let make (variable : variable) (alpha : symbols) (beta : symbols) : t =
    { mVariable = variable; mAlpha = alpha; mBeta = beta }

  let to_string (self : t) : string =
    let alphaStr = String.concat " " self.mAlpha in
    let betaStr  = String.concat " " self.mBeta in
    Printf.sprintf "%s → %s • %s" self.mVariable alphaStr betaStr

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer MarkedRule.pp

Given a *marked rule* `self`, the function `is_complete` checks, whether the *marked rule* `self` has the form
$$ c \rightarrow \alpha\; \bullet,$$
i.e. it checks, whether the $\bullet$ is at the end of the grammar rule.

The Python version adds this function to the class `MarkedRule`.  In OCaml, we *extend* the module `MarkedRule`: the new
module *includes* everything from the old module and adds the function `is_complete`, which is then called as
`MarkedRule.is_complete mr`.

In [ ]:
module MarkedRule = struct
  include MarkedRule

  let is_complete (self : t) : bool = self.mBeta = []
end

Given a *marked rule* `self` of the form
$$ c \rightarrow \alpha \bullet X\, \delta, $$
the function `symbol_after_dot` returns `Some` $X$. If there is no symbol after the $\bullet$, the method returns `None`.

In [ ]:
module MarkedRule = struct
  include MarkedRule

  let symbol_after_dot (self : t) : symbol option =
    match self.mBeta with
    | x :: _ -> Some x
    | [] -> None
end

Given a marked rule of the form
$$ c \rightarrow \alpha \bullet b \delta, $$
this function returns `Some` $b$ if $b$ is a variable.  If there is no variable following the dot, the function returns `None`.

In [ ]:
module MarkedRule = struct
  include MarkedRule

  let next_var (self : t) : variable option =
    match self.mBeta with
    | var :: _ when is_var var -> Some var
    | _ -> None
end

The function `move_dot self` transforms a *marked rule*  of the form 
$$ c \rightarrow \alpha \bullet X\, \beta $$
into a *marked rule* of the form
$$ c \rightarrow \alpha\, X \bullet \beta, $$
i.e. the $\bullet$ is moved over the next symbol.  Invocation of this method assumes that there is a symbol
following the $\bullet$.

In [ ]:
module MarkedRule = struct
  include MarkedRule

  let move_dot (self : t) : t =
    match self.mBeta with
    | x :: rest -> make self.mVariable (self.mAlpha @ [ x ]) rest
    | [] -> invalid_arg "move_dot: there is no symbol after the dot"
end

The function `to_rule self` turns the *marked rule* `self` into a grammar rule, i.e. the *marked rule*
$$ c \rightarrow \alpha \bullet \beta $$
is turned into the grammar rule
$$ c \rightarrow \alpha\, \beta. $$

In [ ]:
module MarkedRule = struct
  include MarkedRule

  let to_rule (self : t) : GrammarRule.t = GrammarRule.make self.mVariable (self.mAlpha @ self.mBeta)
end

## SLR-Table-Generation

The module `Grammar` represents a context free grammar.  Its record type `t` stores the list of the grammar rules of
the given grammar.  Each grammar rule is of the form
$$ a \rightarrow \beta $$
where $\beta$ is a list of variables, tokens, and literals.
The start symbol is assumed to be the variable on the left hand side of the first rule. The grammar is *augmented* with the rule
$$ \widehat{s} \rightarrow s\, \$. $$
Here $s$ is the start variable of the given grammar and $\widehat{s}$ is a new variable that is the start variable of the *augmented grammar*. The symbol `$` denotes the end of input.  The non-obvious fields of the type `Grammar.t` have the following interpretation
- `mStates` is the set of all states of the *SLR-parser*.  These states are sets of *marked rules*.
- `mStateNames` is a dictionary assigning names of the form `s0`, `s1`, $\cdots$, `sn` to the states stored in
  `mStates`.  The functions `action` and `goto` will be defined for *state names*, not for *states*, because
  otherwise the table representing these functions would become both huge and unreadable.
  Therefore, the dictionary `mStateNames` is needed to associate the the states with their names.
- `mConflicts` is a Boolean variable that will be set to `true` if the table generation discovers
  a *shift/reduce conflict* or a *reduce/reduce conflict*.

The fields that are computed later are declared as `mutable`.  An action is either `Shift s`, `Reduce r`, or `Accept`.

The function `make` corresponds to the constructor of the Python class.  It calls the function `compute_tables`, which
is defined later.  As in the Python class, which defines *stubs* of its methods, the module contains *forward
references* for the functions that are defined later.  `!compute_tables_ref` is the function that is currently stored
in the box `compute_tables_ref`.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

type action = Shift of string | Reduce of GrammarRule.t | Accept

type state = MarkedRule.t FSet.t

module Grammar = struct
  type t = {
    mRules       : GrammarRule.t list;
    mStart       : variable;
    mVariables   : variable FSet.t;
    mTokens      : token FSet.t;
    mutable mStates      : state FSet.t;
    mutable mConflicts   : bool;
    mutable mStateNames  : (state, string) Dict.t;
    mutable mRuleNames   : (GrammarRule.t, string) Dict.t;
    mutable mFirst       : (variable, token FSet.t) Dict.t;
    mutable mFollow      : (variable, token FSet.t) Dict.t;
    mutable mActionTable : (string * token, action) Dict.t;
    mutable mGotoTable   : (string * variable, string) Dict.t;
  }

  let compute_tables_ref       : (t -> unit) ref                 = ref not_yet_defined
  let compute_rule_names_ref   : (t -> unit) ref                 = ref not_yet_defined
  let compute_first_ref        : (t -> unit) ref                 = ref not_yet_defined
  let compute_follow_ref       : (t -> unit) ref                 = ref not_yet_defined
  let first_list_ref           : (t -> symbols -> token FSet.t) ref = ref not_yet_defined
  let all_states_ref           : (t -> unit) ref                 = ref not_yet_defined
  let compute_action_table_ref : (t -> unit) ref                 = ref not_yet_defined
  let compute_goto_table_ref   : (t -> unit) ref                 = ref not_yet_defined

  let make (rules : GrammarRule.t list) : t =
    let start = (List.hd rules).mVariable in
    let g = {
      mRules       = rules @ [ GrammarRule.make "ŝ" [ start; "$" ] ];   (* augment the grammar *)
      mStart       = start;
      mVariables   = FSet.add "ŝ" (collect_variables rules);
      mTokens      = FSet.add "$" (collect_tokens rules);                (* short for EOF *)
      mStates      = FSet.empty;
      mConflicts   = false;
      mStateNames  = Dict.empty;
      mRuleNames   = Dict.empty;
      mFirst       = Dict.empty;
      mFollow      = Dict.empty;
      mActionTable = Dict.empty;
      mGotoTable   = Dict.empty;
    } in
    !compute_tables_ref g;
    g
end

Given a set of `variables`, the function `initialize_dictionary` returns a dictionary that assigns the empty set to all variables.
This function is needed to initialize the fields `mFirst` and `mFollow` that are dictionaries storing the *first-set* and
*follow-sets* of the syntactical variables.

In [ ]:
let initialize_dictionary (variables : variable FSet.t) : (variable, token FSet.t) Dict.t =
  FSet.fold (fun a d -> Dict.add a FSet.empty d) variables Dict.empty

Given a `Grammar`, the function `compute_tables` computes
- the sets `First(v)` and `Follow(v)` for every variable `v`,
- the set of all *states* of the *SLR-Parser*,
- the *action table*, and
- the *goto table*. 

Given a grammar `g`,
- the set `g.mFirst` is a dictionary such that `g.mFirst[a] = First(a)` and
- the set `g.mFollow` is a dictionary such that `g.mFollow[a] = Follow(a)` for all variables `a`.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_tables (self : t) : unit =
    self.mFirst  <- initialize_dictionary self.mVariables;
    self.mFollow <- initialize_dictionary self.mVariables;
    !compute_first_ref self;
    !compute_follow_ref self;
    !compute_rule_names_ref self;
    !all_states_ref self;
    !compute_action_table_ref self;
    !compute_goto_table_ref self
end
;;
Grammar.compute_tables_ref := Grammar.compute_tables

The function `compute_rule_names` assigns a unique name to each *rule* of the grammar.  These names are used later
to represent *reduce actions* in the *action table*.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_rule_names (self : t) : unit =
    List.iteri (fun counter rule -> self.mRuleNames <- Dict.add rule ("r" ^ string_of_int counter) self.mRuleNames) self.mRules
end
;;
Grammar.compute_rule_names_ref := Grammar.compute_rule_names

The function `compute_first self` computes the sets $\texttt{First}(c)$ for all variables $c$ and stores them in the dictionary `mFirst`.  Abstractly, given a variable $c$ the function $\texttt{First}(c)$ is the set of all tokens that can start a string that is derived from $c$:
$$\texttt{First}(\texttt{c}) :=
  \Bigl\{ t \in T \Bigm| \exists \gamma \in (V \cup T)^*: \texttt{c} \Rightarrow^* t\,\gamma \Bigr\}.
$$
The definition of the function $\texttt{First}()$ is extended to strings from $(V \cup T)^*$ as follows:
- $\texttt{FirstList}(\lambda) = \{\}$.
- $\texttt{FirstList}(t \beta) = \{ t \}$  if $t \in T$.
- $\texttt{FirstList}(\texttt{a} \beta) = \left\{
       \begin{array}[c]{ll}
         \texttt{First}(\texttt{a}) \cup \texttt{FirstList}(\beta) & \mbox{if $\texttt{a} \Rightarrow^* \lambda$;} \\
         \texttt{First}(\texttt{a})                                & \mbox{otherwise.}
       \end{array}
       \right.
      $

If $\texttt{a}$ is a variable of $G$ and the rules defining $\texttt{a}$ are given as
$$\texttt{a} \rightarrow \alpha_1 \mid \cdots \mid \alpha_n, $$
then we have
$$\texttt{First}(\texttt{a}) = \bigcup\limits_{i=1}^n \texttt{FirstList}(\alpha_i). $$
The dictionary `mFirst` that stores this function is computed via a *fixed point iteration*.  The sets are printed in
the format of Python by the function `python_set` from the file `../lib/nb.ml`.

The functions that compute the tables print a lot of text.  The OCaml kernel of *Jupyter* sends printed text line by
line, and when thousands of lines are printed, it can lag behind, so that some lines show up in the output of the next
cell.  Therefore, these functions collect their output in a *buffer* `out` and show it at the end with the function
`display_text` from the file `../lib/nb.ml`, which sends the whole text at once.  `Printf.bprintf out` works like
`Printf.printf`, but appends to the buffer `out`.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_first (self : t) : unit =
    let out = Buffer.create 4096 in
    let change = ref true in
    while !change do
      change := false;
      List.iter
        (fun (rule : GrammarRule.t) ->
          let a, body = rule.mVariable, rule.mBody in
          let first_body = !first_list_ref self body in
          if not (FSet.subset first_body (Dict.find a self.mFirst)) then begin
            change := true;
            self.mFirst <- Dict.add a (FSet.union (Dict.find a self.mFirst) first_body) self.mFirst
          end)
        self.mRules
    done;
    Buffer.add_string out ("First sets:" ^ "\n");
    FSet.iter
      (fun v -> Printf.bprintf out "First(%s) = %s\n" v (python_set python_repr (FSet.to_list (Dict.find v self.mFirst))))
      self.mVariables;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_first_ref := Grammar.compute_first

Given a list of variables and tokens `alpha`, the function `first_list self alpha` computes the function $\texttt{FirstList}(\alpha)$ that has been defined above.  If `alpha` is *nullable*, then the result will contain the empty string $\lambda$.
The function `eps_union` is defined below, so it is called via a forward reference, too.

In [ ]:
let eps_union_ref : (token FSet.t -> token FSet.t -> token FSet.t) ref = ref not_yet_defined

module Grammar = struct
  include Grammar

  let rec first_list (self : t) (alpha : symbols) : token FSet.t =
    match alpha with
    | [] -> FSet.singleton ""
    | v :: r when is_var v -> !eps_union_ref (Dict.find v self.mFirst) (first_list self r)
    | t :: _ -> FSet.singleton t
end
;;
Grammar.first_list_ref := Grammar.first_list

The arguments `s` and `t` of `eps_union` are sets that contain tokens and, additionally, they might contain the empty string
$\lambda =$ `""`.  The specification of `eps_union` is:
$$ \mathtt{eps\_union}(S, T) = \left\{ \begin{array}{ll}
                                       S          & \mbox{if $\lambda \not\in S$} \\
                                       S \cup T   & \mbox{if $\lambda \in S \wedge \lambda \in T$} \\
                                       S \cup T - \{\lambda \} & \mbox{if $\lambda \in S \wedge \lambda \not\in T$}
                                      \end{array}
                              \right.
$$

In [ ]:
let eps_union (s : token FSet.t) (t : token FSet.t) : token FSet.t =
  if FSet.mem "" s then
    if FSet.mem "" t then FSet.union s t
    else FSet.union (FSet.remove "" s) t
  else s
;;
eps_union_ref := eps_union

Given an augmented grammar $G = \langle V,T,R\cup\{\widehat{s} \rightarrow s\,\$\}, \widehat{s}\rangle$ 
and a variable $a$, the set of tokens that might follow $a$ is defined as:
$$\texttt{Follow}(a) := 
 \bigl\{ t \in \widehat{T} \,\bigm|\, \exists \beta,\gamma \in (V \cup \widehat{T})^*: 
                           \widehat{s} \Rightarrow^* \beta \,a\, t\, \gamma 
  \bigr\}.
$$
The function `compute_follow` computes the sets $\texttt{Follow}(a)$ for all variables $a$ via a *fixed-point iteration*.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_follow (self : t) : unit =
    let out = Buffer.create 4096 in
    self.mFollow <- Dict.add self.mStart (FSet.singleton "$") self.mFollow;
    let change = ref true in
    while !change do
      change := false;
      List.iter
        (fun (rule : GrammarRule.t) ->
          let a, body = rule.mVariable, rule.mBody in
          List.iteri
            (fun i yi ->
              if is_var yi then begin
                let tail      = first_list self (List.filteri (fun j _ -> j > i) body) in
                let firstTail = eps_union tail (Dict.find a self.mFollow) in
                if not (FSet.subset firstTail (Dict.find yi self.mFollow)) then begin
                  change := true;
                  self.mFollow <- Dict.add yi (FSet.union (Dict.find yi self.mFollow) firstTail) self.mFollow
                end
              end)
            body)
        self.mRules
    done;
    Buffer.add_string out "Follow sets (note that \"$\" denotes the end of file):\n";
    FSet.iter
      (fun v -> Printf.bprintf out "Follow(%s) = %s\n" v (python_set python_repr (FSet.to_list (Dict.find v self.mFollow))))
      self.mVariables;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_follow_ref := Grammar.compute_follow

If $\mathcal{M}$ is a set of *marked rules*, then the *closure* of $\mathcal{M}$ is the smallest set $\mathcal{K}$ such that
we have the following:
- $\mathcal{M} \subseteq \mathcal{K}$,
- If $a \rightarrow \beta \bullet c\, \delta$ is a *marked rule* from 
  $\mathcal{K}$, and $c$ is a variable and if, furthermore,
  $c \rightarrow \gamma$ is a grammar rule,
  then the marked rule $c \rightarrow \bullet \gamma$
  is an element of $\mathcal{K}$:
  $$(a \rightarrow \beta \bullet c\, \delta) \in \mathcal{K} 
         \;\wedge\; 
         (c \rightarrow \gamma) \in R
         \;\Rightarrow\; (c \rightarrow \bullet \gamma) \in \mathcal{K}
  $$

We define $\texttt{closure}(\mathcal{M}) := \mathcal{K}$.  The function `cmp_closure` computes this closure for a given set of *marked rules* via a *fixed-point iteration*.

In [ ]:
module Grammar = struct
  include Grammar

  let cmp_closure (self : t) (marked_Rules : state) : state =
    let rec loop (all_Rules : state) (new_Rules : state) : state =
      let more_Rules =
        let open FSet in
        let* rule = new_Rules in
        match MarkedRule.next_var rule with
        | None -> empty
        | Some c ->
            of_list
              (List.filter_map
                 (fun (r : GrammarRule.t) -> if c = r.mVariable then Some (MarkedRule.make r.mVariable [] r.mBody) else None)
                 self.mRules)
      in
      if FSet.subset more_Rules all_Rules then all_Rules
      else
        let new_Rules = FSet.diff more_Rules all_Rules in
        loop (FSet.union all_Rules new_Rules) new_Rules
    in
    loop marked_Rules marked_Rules
end

Given a set of *marked rules* $\mathcal{M}$ and a *grammar symbol* $X$, the function $\texttt{goto}(\mathcal{M}, X)$ 
is defined as follows:
$$\texttt{goto}(\mathcal{M}, X) := \texttt{closure}\Bigl( \bigl\{ 
   a \rightarrow \beta\, X \bullet \delta \bigm| (a \rightarrow \beta \bullet X\, \delta) \in \mathcal{M} 
   \bigr\} \Bigr).
$$

In [ ]:
module Grammar = struct
  include Grammar

  let goto (self : t) (marked_Rules : state) (x : symbol) : state =
    cmp_closure self
      (FSet.map MarkedRule.move_dot (FSet.filter (fun mr -> MarkedRule.symbol_after_dot mr = Some x) marked_Rules))
end

The function `all_states` computes the set of all states of an *SLR-parser*.  The function starts with the state
$$ \texttt{closure}\bigl(\{ \widehat{s} \rightarrow \bullet s \, $\}\bigr) $$
and then tries to compute new states by using the function `goto`.  This computation proceeds via a
*fixed-point iteration*.  Once all states have been computed, the function assigns names to these states.
This association is stored in the dictionary `mStateNames`.  The function `state_to_string` prints a state like
Python prints a set of marked rules.

In [ ]:
let state_to_string (state : state) : string = python_set MarkedRule.to_string (FSet.to_list state)

module Grammar = struct
  include Grammar

  let all_states (self : t) : unit =
    let out = Buffer.create 4096 in
    let start_state = cmp_closure self (FSet.singleton (MarkedRule.make "ŝ" [] [ self.mStart; "$" ])) in
    self.mStates <- FSet.singleton start_state;
    let rec loop (new_States : state FSet.t) =
      let more_States =
        let open FSet in
        let* rule_Set = new_States in
        let* mr = rule_Set in
        if MarkedRule.is_complete mr then empty
        else
          match MarkedRule.symbol_after_dot mr with
          | Some x when x <> "$" -> return (goto self rule_Set x)
          | _ -> empty
      in
      if not (FSet.subset more_States self.mStates) then begin
        let new_States = FSet.diff more_States self.mStates in
        self.mStates <- FSet.union self.mStates new_States;
        loop new_States
      end
    in
    loop self.mStates;
    Buffer.add_string out ("All SLR-states:" ^ "\n");
    self.mStateNames <- Dict.add start_state "s0" self.mStateNames;
    Printf.bprintf out "s0 = %s\n" (state_to_string start_state);
    let counter = ref 1 in
    FSet.iter
      (fun state ->
        self.mStateNames <- Dict.add state (Printf.sprintf "s%d" !counter) self.mStateNames;
        Printf.bprintf out "s%d = %s\n" !counter (state_to_string state);
        incr counter)
      (FSet.remove start_state self.mStates);
    display_text (Buffer.contents out)
end
;;
Grammar.all_states_ref := Grammar.all_states

The following function computes the *action table* and is defined as follows:
- If $\mathcal{M}$ contains a *marked rule* of the form $a \rightarrow \beta \bullet t\, \delta$
  then we have
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{shift}, \texttt{goto}(\mathcal{M},t) \rangle.$$
- If $\mathcal{M}$ contains a marked rule of the form $a \rightarrow \beta\, \bullet$ and we have
  $t \in \texttt{Follow}(a)$, then we define
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{reduce}, a \rightarrow \beta \rangle$$
- If $\mathcal{M}$ contains the marked rule $\widehat{s} \rightarrow s \bullet \$ $, then we define
  $$\texttt{action}(\mathcal{M},\$) := \texttt{accept}. $$
- Otherwise, we have
  $$\texttt{action}(\mathcal{M},t) := \texttt{error}. $$

The function `action_to_string` prints an action like Python prints the corresponding tuple.

In [ ]:
let action_to_string (action : action) : string =
  match action with
  | Shift s -> Printf.sprintf "('shift', '%s')" s
  | Reduce r -> Printf.sprintf "('reduce', %s)" (GrammarRule.to_string r)
  | Accept -> "'accept'"

module Grammar = struct
  include Grammar

  let compute_action_table (self : t) : unit =
    let out = Buffer.create 4096 in
    self.mActionTable <- Dict.empty;
    Buffer.add_string out ("\nAction Table:" ^ "\n");
    FSet.iter
      (fun state ->
        let stateName = Dict.find state self.mStateNames in
        let actionTable = ref Dict.empty in
        (* compute shift actions *)
        FSet.iter
          (fun token ->
            if token <> "$" then begin
              let newState = goto self state token in
              if not (FSet.is_empty newState) then begin
                let newName = Dict.find newState self.mStateNames in
                actionTable := Dict.add token (Shift newName) !actionTable;
                self.mActionTable <- Dict.add (stateName, token) (Shift newName) self.mActionTable;
                Printf.bprintf out "action(\"%s\", %s) = (\"shift\", %s)\n" stateName token newName
              end
            end)
          self.mTokens;
        (* compute reduce actions *)
        FSet.iter
          (fun mr ->
            if MarkedRule.is_complete mr then
              FSet.iter
                (fun token ->
                  let action2 = Reduce (MarkedRule.to_rule mr) in
                  match Dict.find_opt token !actionTable with
                  | None ->
                      actionTable := Dict.add token action2 !actionTable;
                      self.mActionTable <- Dict.add (stateName, token) action2 self.mActionTable;
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action2)
                  | Some action1 when action1 <> action2 ->
                      self.mConflicts <- true;
                      Buffer.add_string out "\n";
                      Printf.bprintf out "conflict in state %s:\n" stateName;
                      Printf.bprintf out "%s = %s\n" stateName (state_to_string state);
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action1);
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action2);
                      Buffer.add_string out "\n"
                  | Some _ -> ())
                (Dict.find mr.MarkedRule.mVariable self.mFollow))
          state;
        FSet.iter
          (fun mr ->
            if mr = MarkedRule.make "ŝ" [ self.mStart ] [ "$" ] then begin
              actionTable := Dict.add "$" Accept !actionTable;
              self.mActionTable <- Dict.add (stateName, "$") Accept self.mActionTable;
              Printf.bprintf out "action(\"%s\", $) = accept\n" stateName
            end)
          state)
      self.mStates;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_action_table_ref := Grammar.compute_action_table

The function `compute_goto_table` computes the *goto table*.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_goto_table (self : t) : unit =
    let out = Buffer.create 4096 in
    self.mGotoTable <- Dict.empty;
    Buffer.add_string out ("\nGoto Table:" ^ "\n");
    FSet.iter
      (fun state ->
        FSet.iter
          (fun var ->
            let newState = goto self state var in
            if not (FSet.is_empty newState) then begin
              let stateName = Dict.find state self.mStateNames in
              let newName   = Dict.find newState self.mStateNames in
              self.mGotoTable <- Dict.add (stateName, var) newName self.mGotoTable;
              Printf.bprintf out "goto(%s, %s) = %s\n" stateName var newName
            end)
          self.mVariables)
      self.mStates;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_goto_table_ref := Grammar.compute_goto_table

In [ ]:
grammar

The function `time` from the file `../lib/nb.ml` measures the time needed to compute the tables, like the magic
command `%%time`.

In [ ]:
let g = time (fun () -> Grammar.make grammar)

The function `dump_parse_table` writes the parse table to a file.  The Python version writes Python code to the file
`parse-table.py`.  The OCaml version writes OCaml code in the style of the notebook `Parse-Table.ipynb` to the file
`parse-table.ml`: the rules are pairs, the action table and the goto table are filled via the index operators `.%{ }`,
and the states are stored as strings in the list `stateNames`.  The function `strip_quotes` removes the quotes from a literal.

In [ ]:
let strip_quotes (t : string) : string =
  let n = String.length t in
  if n >= 2 && t.[0] = '\'' && t.[n - 1] = '\'' then String.sub t 1 (n - 2) else t

In [ ]:
module Grammar = struct
  include Grammar

  let dump_parse_table (self : t) (file : string) : unit =
    let ml_list l = "[" ^ String.concat "; " (List.map repr l) ^ "]" in
    Out_channel.with_open_text file (fun handle ->
        let write s = output_string handle s in
        write "(* Grammar rules: *)\n";
        List.iter
          (fun (rule : GrammarRule.t) ->
            let rule_name = Dict.find rule self.mRuleNames in
            write (Printf.sprintf "let %s = (%s, %s)\n" rule_name (repr rule.mVariable) (ml_list rule.mBody)))
          self.mRules;
        write "\n(* Action table: *)\n";
        write "let actionTable : (string * string, action) Hashtbl.t = Hashtbl.create 64;;\n";
        Dict.iter
          (fun (s, t) action ->
            let t = strip_quotes t in
            match action with
            | Reduce rule ->
                write (Printf.sprintf "actionTable.%%{%s, %s} <- Reduce %s;;\n" (repr s) (repr t) (Dict.find rule self.mRuleNames))
            | Accept -> write (Printf.sprintf "actionTable.%%{%s, %s} <- Accept;;\n" (repr s) (repr t))
            | Shift s2 -> write (Printf.sprintf "actionTable.%%{%s, %s} <- Shift %s;;\n" (repr s) (repr t) (repr s2)))
          self.mActionTable;
        write "\n(* Goto table: *)\n";
        write "let gotoTable : (string * string, string) Hashtbl.t = Hashtbl.create 64;;\n";
        Dict.iter
          (fun (s, v) state -> write (Printf.sprintf "gotoTable.%%{%s, %s} <- %s;;\n" (repr s) (repr v) (repr state)))
          self.mGotoTable;
        write "\n(* States: *)\n";
        write "let stateNames = [\n";
        Dict.iter (fun s name -> write (Printf.sprintf "  (%s, {|%s|});\n" (repr name) (state_to_string s))) self.mStateNames;
        write "]\n")
end

In [ ]:
Grammar.dump_parse_table g "parse-table.ml"

In [ ]:
cat "parse-table.ml"

In [ ]:
let analyse (file : string) : unit =
  let rules        = parse file in
  let grammarRules = transform rules in
  let grammar      = Grammar.make grammarRules in
  Grammar.dump_parse_table grammar "parse-table.ml"

In [ ]:
analyse "Examples/cc.g"

In [ ]:
analyse "Examples/postfix.g"

In [ ]:
analyse "Examples/arith-small.g"

In [ ]:
time (fun () -> analyse "Examples/arith-ambiguous.g")

In [ ]:
time (fun () -> analyse "Examples/c-grammar-slr.g")

In [ ]:
time (fun () -> analyse "Examples/c-grammar.g")

The file `Examples/bool-2.g` does not exist, so the last cell raises an exception, as it does in the Python version.

In [ ]:
analyse "Examples/bool-2.g"